In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from pathlib import Path

# Imports du projet
import sys
sys.path.append(str(Path.cwd().parent))
from mpp_project.core import load_tournament_data
from mpp_project.v5_supervised.oracle_dp import evaluate_match_ensemble 

# --- CONFIGURATION ---
MATCH_DU_JOUR_ID = 1    
MON_GAP_1 = -300
MON_GAP_2 = -300
HAS_BOOSTER = 1  # NOUVEAU : 1 = Booster disponible, 0 = Booster déjà utilisé

# 1. Chargements des données
notebook_dir = Path.cwd()
project_root = notebook_dir.parent 
data_path_csv = project_root / "data" / "CDM_2026_imputed.csv"
data_path_npy = project_root / "data" / "expected_V_phases_finales.npy"

df, true_probas, mpp_gains, crowd_repartitions = load_tournament_data(data_path_csv)
V_all_matches = np.load(data_path_npy)

match_phases = df['phase'].tolist()
match_idx = MATCH_DU_JOUR_ID - 1
last_csv_match_id = df['match_id'].max()

# Extraction de l'horizon
nb_poules = len(df[df['phase'].str.contains('Poule', na=False)]) 
npy_index = last_csv_match_id - nb_poules
V_next_base = V_all_matches[npy_index] # Contient désormais la dimension 4D [..., 2]

# --- 2. APPEL DE L'ORACLE ---
print("Consultation de l'Oracle (Super-Adversaire + Booster x2)...\n")

# L'appel à l'oracle est mis à jour pour récupérer les deux vecteurs de Win Rate
wr_keep, wr_use, ev_final = evaluate_match_ensemble(
    match_idx=match_idx, 
    last_csv_match_id=last_csv_match_id,
    true_probas=true_probas, 
    match_phases=match_phases, 
    mpp_gains=mpp_gains,
    V_next_base=V_next_base, 
    gap1=MON_GAP_1, 
    gap2=MON_GAP_2,
    has_booster=HAS_BOOSTER,
    n_ensembles=30
)

# --- 3. AFFICHAGE DE LA DÉCISION ---
print(f"--- MATCH {MATCH_DU_JOUR_ID} : ANALYSE DES CHOIX ---")
noms_choix = ["1 (Favori)", "N (Nul)", "2 (Outsider)"]

if HAS_BOOSTER:
    for i in range(3):
        print(f"Choix {noms_choix[i]} :")
        print(f"  -> WR (Garder x2)   = {wr_keep[i]*100:.2f}%")
        print(f"  -> WR (Utiliser x2) = {wr_use[i]*100:.2f}% | EV (x2) = {ev_final[i]*2:.2f} pts")
    
    best_keep_idx = np.argmax(wr_keep)
    best_use_idx = np.argmax(wr_use)
    
    if wr_use[best_use_idx] > wr_keep[best_keep_idx]:
        print(f"\n🚀 RECOMMANDATION ROBUSTE : Jouer le {noms_choix[best_use_idx]} ET UTILISER LE BOOSTER x2 (Gain WR: +{(wr_use[best_use_idx] - wr_keep[best_keep_idx])*100:.2f}%)")
    else:
        print(f"\n✅ RECOMMANDATION ROBUSTE : Jouer le {noms_choix[best_keep_idx]} (Garder le Booster pour plus tard)")
else:
    for i in range(3):
        print(f"Choix {noms_choix[i]} : Win Rate = {wr_keep[i]*100:.2f}% | EV = {ev_final[i]:.2f} pts")
    best_action = np.argmax(wr_keep)
    print(f"\n✅ RECOMMANDATION ROBUSTE : Jouer le {noms_choix[best_action]}")

plan_one_match_ahead = False
if plan_one_match_ahead:
    # --- MODE NUIT : ANTICIPATION DU MATCH SUIVANT ---
    MATCH_DE_LA_NUIT_ID = MATCH_DU_JOUR_ID + 1
    match_nuit_idx = MATCH_DE_LA_NUIT_ID - 1

    print(f"\n🌙 --- MODE NUIT : SIMULATION POUR LE MATCH {MATCH_DE_LA_NUIT_ID} ---")

    # On estime le gain moyen du Match en cours à environ 35 points
    swing_points = 35 

    # On crée nos 3 univers possibles après le Match 1
    univers_gaps = {
        "Pire Cas (-35 pts)": (MON_GAP_1 - swing_points, MON_GAP_2 - swing_points),
        "Statu Quo (0 pts) ": (MON_GAP_1, MON_GAP_2),
        "Meilleur Cas (+35)": (MON_GAP_1 + swing_points, MON_GAP_2 + swing_points)
    }

    for nom_univers, (hypo_gap1, hypo_gap2) in univers_gaps.items():
        wr_keep, wr_use, ev_final = evaluate_match_ensemble(
            match_idx=match_nuit_idx, 
            last_csv_match_id=last_csv_match_id,
            true_probas=true_probas, 
            match_phases=match_phases, 
            mpp_gains=mpp_gains,
            V_next_base=V_next_base, 
            gap1=hypo_gap1, 
            gap2=hypo_gap2,
            has_booster=HAS_BOOSTER,
            n_ensembles=15 # Un peu moins d'ensembles pour aller vite
        )
        
        best_action_idx = np.argmax(wr_keep)
        print(f"{nom_univers} -> Oracle recommande : Jouer le {noms_choix[best_action_idx]} (WR: {wr_keep[best_action_idx]*100:.2f}%)")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Consultation de l'Oracle (Super-Adversaire + Booster x2)...

--- MATCH 1 : ANALYSE DES CHOIX ---
Choix 1 (Favori) :
  -> WR (Garder x2)   = 31.88%
  -> WR (Utiliser x2) = 26.24% | EV (x2) = 61.71 pts
Choix N (Nul) :
  -> WR (Garder x2)   = 31.95%
  -> WR (Utiliser x2) = 26.61% | EV (x2) = 57.59 pts
Choix 2 (Outsider) :
  -> WR (Garder x2)   = 31.89%
  -> WR (Utiliser x2) = 26.49% | EV (x2) = 41.42 pts

✅ RECOMMANDATION ROBUSTE : Jouer le N (Nul) (Garder le Booster pour plus tard)
